# Type validation: in the editor while you type, and when the cell runs

In C#, giving a text to a number property is a compile error. In plain Python it would go through, and the error would turn up later, far from its cause. The classes of `bhom-schema` are generated from the BHoM C# classes **with their types**, so a wrong value is caught in **two places**:

| | When | What you see | What it needs |
| --- | --- | --- | --- |
| **1. The editor** | while you type, before anything runs | a red underline on the wrong value, with a message on hover | VS Code with the Python (Pylance) extension, or PyCharm |
| **2. The class** | when the cell runs | a `ValidationError` that names every wrong field, with the path to it and the allowed values | nothing: it is part of the class, in a notebook, a script or a Databricks job |

This notebook goes through the mistakes one at a time. Each section says what the editor shows (look for the underline) and what the cell raises when you run it.

**How to read the cells**

- 🟢 **runs without an error.**
- 🔴 **raises an exception when you run it. That is the point of the cell.** Run these cells one at a time (*Shift+Enter*); *Run All* stops at the first one. The exception comes out in red under the cell, and it is the first thing to look at.
- 🟡 **the editor and the run do not agree.** These show where each check has limits.

**Set the editor up once (VS Code).** Pylance only underlines wrong types when its type checking is on. In *Settings*, search for `python.analysis.typeCheckingMode` and set it to `basic` (the default, `off`, shows no type errors). Select, as the notebook kernel, the Python environment where `bhom-schema` is installed (the next cell installs it). If an import is underlined right after the install, run *Python: Restart Language Server* from the command palette.

## 1. Setup

The first cell installs or updates the package into the kernel's environment (skip it if `bhom-schema` is already installed; on Databricks it needs a restart of the Python process afterwards: see `Databricks-DataSchema-Demo.ipynb`). The second imports the classes. **Expect:** the package version, then no errors, and no underline on any import.

In [ ]:
%pip install --quiet --upgrade --pre --no-cache-dir --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ bhom-schema

In [ ]:
import bhom_schema
from pydantic import ValidationError

from bhom_schema.DataSchema_oM.Building import Building
from bhom_schema.DataSchema_oM.Elements.Beam import Beam
from bhom_schema.DataSchema_oM.Enums.SlabType import SlabType
from bhom_schema.DataSchema_oM.Enums.StructuralMaterial import StructuralMaterial
from bhom_schema.DataSchema_oM.ExampleObject import ExampleObject
from bhom_schema.DataSchema_oM.Grid import Grid
from bhom_schema.DataSchema_oM.Storey import Storey

print("bhom-schema", bhom_schema.__version__)

## 2. 🟢 Valid objects

Start with correct code, to see what the editor does **not** underline. Type `Storey(` in a new cell: the editor lists the parameters, named like the C# properties, each with its type (`TypicalHeight: float | None`); `Building(` and `Beam(` the same. **Expect:** no underline anywhere, and the printed values.

In [ ]:
storey = Storey(Name="Typical storey", TypicalHeight=3.9, NonRoofArea=1850.0)
building = Building(Name="Riverside Tower", StoreysAboveGround=24, MaterialVolumesAboveGround={"Concrete": 9800.0})
beam = Beam(TypicalMaterial=StructuralMaterial.SteelHotRolled, TypicalCountPerSlab=48)

print(storey.TypicalHeight, building.StoreysAboveGround, beam.TypicalMaterial)

## 3. 🔴 A text where a number is expected

`TypicalHeight` is a number (`float`, in metres, as in C#).

- **Editor:** the text `"tall"` is underlined: *Argument of type "Literal['tall']" cannot be assigned to parameter "TypicalHeight" of type "float | None"*.
- **Run:** `ValidationError: 1 validation error for Storey`, then `TypicalHeight` and *Input should be a valid number, unable to parse string as a number*.

In [ ]:
Storey(TypicalHeight="tall")

## 4. 🔴 A fractional number where a whole number is expected

`StoreysAboveGround` is an `int`, as in C#: 12.5 storeys is not a thing.

- **Editor:** `12.5` is underlined: *Argument of type "float" cannot be assigned to parameter "StoreysAboveGround" of type "int | None"*.
- **Run:** *Input should be a valid integer, got a number with a fractional part*.

In [ ]:
Building(StoreysAboveGround=12.5)

## 5. 🔴 A value that is not a member of an enum

`TypicalMaterial` is the C# enum `StructuralMaterial`, a real Python enum. Type `StructuralMaterial.` and the editor lists the members.

- **Editor:** the text is underlined: *Argument of type "Literal['Unobtainium']" cannot be assigned to parameter "TypicalMaterial" of type "StructuralMaterial | None"*.
- **Run:** the message names the wrong value and **lists every member**: *Unobtainium is not a member of StructuralMaterial. Members: SteelHotRolled, SteelLightGauge, Composite, ...*.

In [ ]:
Beam(TypicalMaterial="Unobtainium")

## 6. 🔴 A member of the wrong enum

`SlabType.CompositeMetalDeck` is a real member of a real enum, but of another one than `Beam.TypicalMaterial` takes.

- **Editor:** *Argument of type "Literal[SlabType.CompositeMetalDeck]" cannot be assigned to parameter "TypicalMaterial" of type "StructuralMaterial | None"*.
- **Run:** *expected a member name of StructuralMaterial*.

In [ ]:
Beam(TypicalMaterial=SlabType.CompositeMetalDeck)

## 7. 🔴 A wrong value inside a dictionary

`MaterialVolumesAboveGround` is a dictionary from material name to volume (`dict[str, float]`).

- **Editor:** the dictionary is underlined: *Argument of type "dict[str, str]" cannot be assigned to parameter "MaterialVolumesAboveGround" of type "dict[str, float] | None"*.
- **Run:** the message gives the **path to the wrong item**, `MaterialVolumesAboveGround.Concrete`, and *Input should be a valid number*.

In [ ]:
Building(MaterialVolumesAboveGround={"Concrete": "lots"})

## 8. 🔴 An object of another class

`ExampleObject.MainGrid` holds a `Grid`, and `Storeys` a list of `Storey`. A `Storey` is not a `Grid`, even if it has the same base class.

- **Editor:** *Argument of type "Storey" cannot be assigned to parameter "MainGrid" of type "Grid | None"*.
- **Run:** *Input should be a valid dictionary or instance of Grid*, on the field `MainGrid`.

In [ ]:
ExampleObject(MainGrid=Storey())

The same for one item of a list.

- **Editor:** the list is underlined: *Argument of type "list[Storey | Grid]" cannot be assigned to parameter "Storeys" of type "list[Storey]"*.
- **Run:** the path points to the item, `Storeys.1` (the second), and the message is the same as above, for `Storey`.

In [ ]:
ExampleObject(Storeys=[Storey(TypicalHeight=3.5), Grid()])

## 9. 🔴 Changing a value afterwards

The check does not only run when the object is created. It runs again every time a field is assigned, so a notebook cell such as `storey.TypicalHeight = "tall"` cannot put a bad value into an object that was good.

- **Editor:** *Cannot assign to attribute "TypicalHeight" for class "Storey"*.
- **Run:** the same `ValidationError` as in section 3.

In [ ]:
storey.TypicalHeight = "tall"

The refused assignment changed nothing. **Expect:** `3.9`, the value of section 2.

In [ ]:
print(storey.TypicalHeight)

## 10. 🔴 A misspelt field name

- **Editor:** *Attribute "TypicalHeigth" is unknown*.
- **Run:** `ValidationError`: *Object has no attribute 'TypicalHeigth'*.

In [ ]:
storey.TypicalHeigth = 3.0

## 11. 🟡 The same typo, in the keyword of a constructor

Only **one** of the two checks catches this.

- **Editor:** *No parameter named "TypicalHeigth"* is underlined, as above.
- **Run:** **nothing is raised.** An unknown keyword in the constructor is ignored, so the object is created without the height. The cell prints `None`.

This is the one place where the editor sees more than the run does. If code with such a typo runs outside an editor (a script, a Databricks job), the value is silently dropped.

In [ ]:
misspelt = Storey(Name="Typical storey", TypicalHeigth=3.5)
print(misspelt.TypicalHeight)

## 12. 🟡 The editor is stricter than the run

Data comes from files, tables and web forms as text, so a value that **reads as** the right type is accepted at run time and converted. The editor only sees the type of what you wrote, so it underlines these two lines.

- `TypicalHeight="3.5"`: the text `3.5` becomes the number 3.5.
- `TypicalMaterial="SteelHotRolled"`: the C# name of a member becomes the member.

**Expect:** both lines underlined, no error, and the converted values. To avoid the underline, give the real type: a number, a `StructuralMaterial` member.

In [ ]:
converted = Storey(TypicalHeight="3.5")
named = Beam(TypicalMaterial="SteelHotRolled")

print(repr(converted.TypicalHeight), repr(named.TypicalMaterial))

## 13. 🔴 All the errors at once

The message does not stop at the first mistake. This call has two wrong values and one right one.

- **Editor:** both wrong values are underlined.
- **Run:** `2 validation errors for Storey`, one for `TypicalHeight` and one for `NonRoofArea`; the correct `RoofArea` is not mentioned.

In [ ]:
Storey(TypicalHeight="tall", NonRoofArea="big", RoofArea=2.0)

## 14. 🟢 Catch the exception and read it

In a script you decide what happens to a refused value. `ValidationError` lists the errors as data: the path to the field, the message and the value that was given. **Expect:** `2 errors`, then one line for each.

In [ ]:
try:
    Storey(TypicalHeight="tall", NonRoofArea="big", RoofArea=2.0)
except ValidationError as error:
    print(f"{error.error_count()} errors")
    for item in error.errors():
        field = ".".join(str(part) for part in item["loc"])
        print(f"  {field}: {item['msg']} (given: {item['input']!r})")

## 15. 🟢 Data from JSON is checked as it is read

A JSON text, or a dictionary, from a file or a table has no Python type that the editor could see, so **only the run checks it**, and that is where most real mistakes come in. The first text is valid BHoM JSON. **Expect:** the building, with its type name (`_t`) written the way BHoM writes it.

In [ ]:
valid_json = '{"_t": "BH.oM.DataSchema.Building", "Name": "Harbour Library", "StoreysAboveGround": 5, "MaterialVolumesAboveGround": {"Timber": 1250.0}}'
loaded = Building.model_validate_json(valid_json)

print(loaded.Name, loaded.StoreysAboveGround, loaded.MaterialVolumesAboveGround)
print(loaded.model_dump_json()[:90], "...")

## 16. 🔴 Wrong JSON

The same text with two wrong values. There is **no underline**: the editor cannot see inside a text. **Run:** `2 validation errors for Building`, with the path to each (`StoreysAboveGround`, `MaterialVolumesAboveGround.Steel`).

In [ ]:
wrong_json = '{"_t": "BH.oM.DataSchema.Building", "Name": "Harbour Library", "StoreysAboveGround": "many", "MaterialVolumesAboveGround": {"Steel": "lots"}}'
Building.model_validate_json(wrong_json)

## 17. What catches what

| Mistake | Editor, while typing | Run, as an exception |
| --- | :---: | :---: |
| Wrong type for a field (text for a number) | ✅ | ✅ |
| Fractional number for a whole number | ✅ | ✅ |
| Value that is not a member of the enum, or a member of another enum | ✅ | ✅ (lists the members) |
| Wrong item inside a list or a dictionary | ✅ | ✅ (path to the item) |
| An object of another class | ✅ | ✅ |
| Assigning a wrong value later | ✅ | ✅ |
| Assigning to a misspelt field | ✅ | ✅ |
| Misspelt keyword in the constructor | ✅ | ❌ ignored |
| Text that reads as the right type (`"3.5"`, `"SteelHotRolled"`) | underlined | accepted, converted |
| Wrong value inside JSON, a dictionary or table rows | ❌ cannot see | ✅ (all errors at once) |

The notebook `Databricks-DataSchema-Demo.ipynb` shows the third place where a value is checked: the Delta table itself, which can refuse a wrong enum value from any writer.